In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
#retstart runtime after

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu!!")

CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q sentence-transformers

### Upload Files

In [3]:
from google.colab import files
uploaded = files.upload()

Saving processed_documents.zip to processed_documents.zip


In [4]:
import zipfile
with zipfile.ZipFile("processed_documents.zip") as z:
    z.extractall("data")
# adjust if your zip nests differently — check with the next line
!find data -name "metadata.json"

data/processed_documents/metadata.json


### loader

In [5]:
import json
from pathlib import Path

PROCESSED = Path("data/processed_documents")   # match what `find` showed

def load_corpus():
    master = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    corpus, skipped = [], 0
    for fname, meta in master.items():
        if meta.get("is_duplicate"):
            skipped += 1
            continue
        stem = Path(fname).stem
        dj = PROCESSED / f"{stem}.json"
        if not dj.exists():
            continue
        data = json.loads(dj.read_text(encoding="utf-8"))
        for c in data["chunks"]["chunks"]:
            t = c["text"].strip()
            if not t:
                continue
            corpus.append({
                "chunk_id": f"{stem}__{c['chunk_index']}",
                "text": t, "source_doc": fname,
                "language": meta.get("primary_language"),
                "token_count": c.get("token_count"),
            })
    print(f"[loader] {len(corpus)} chunks | {skipped} dupe docs skipped")
    return corpus

corpus = load_corpus()

[loader] 10456 chunks | 12 dupe docs skipped


### embed function

In [6]:
import numpy as np, json, time
from sentence_transformers import SentenceTransformer
from google.colab import files

def embed_and_download(model_name, hf_id, prefix="", batch_size=64):
    print(f"\n=== {model_name} ===")
    model = SentenceTransformer(hf_id, device="cuda")

    texts = [prefix + c["text"] for c in corpus]   # prefix applied here
    ids   = [c["chunk_id"] for c in corpus]

    t = time.time()
    vecs = model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True)
    dt = time.time() - t
    vecs = vecs.astype(np.float32)

    np.save(f"{model_name}.npy", vecs)
    with open(f"{model_name}.meta.json", "w") as f:
        json.dump({"model": model_name, "hf_id": hf_id, "prefix": prefix,
                   "dim": int(vecs.shape[1]), "count": int(vecs.shape[0]),
                   "chunk_ids": ids, "encode_seconds": round(dt, 1)}, f)

    print(f"done: {vecs.shape} in {dt:.1f}s ({1000*dt/len(texts):.0f} ms/chunk)")
    files.download(f"{model_name}.npy")
    files.download(f"{model_name}.meta.json")

## BGE
english baseline

In [7]:
embed_and_download("bge-large", "BAAI/bge-large-en-v1.5", prefix="")


=== bge-large ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

done: (10456, 1024) in 1055.3s (101 ms/chunk)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## BGE-M3
multilingual, no prefix, big context window

most versatile and advanced model in this list !

In [10]:
embed_and_download("bge-m3", "BAAI/bge-m3", prefix="")


=== bge-m3 ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

done: (10456, 1024) in 975.3s (93 ms/chunk)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## E5
learn to pull similar texts together and push dissimilar ones apart in the vector space, requires prefix

In [8]:
embed_and_download("e5-large", "intfloat/e5-large-v2", prefix="passage: ")


=== e5-large ===


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

...

done: (10456, 1024) in 1074.7s (103 ms/chunk)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Multilingual E5

In [9]:
embed_and_download("multilingual-e5-large", "intfloat/multilingual-e5-large", prefix="passage: ")


=== multilingual-e5-large ===


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

done: (10456, 1024) in 879.6s (84 ms/chunk)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>